In [6]:
import numpy as np

**INITIALIZATION**

In [42]:
def function(x1,x2):
    return (x1 + x2 - 7)**2 + (2*x1 + 3*x2 - 6)**2 - 7

**SOLUTION**

In [43]:

def function(x1, x2):
    return (x1 + x2 - 7)**2 + (2*x1 + 3*x2 - 6)**2 - 7

def genetic_algorithm(
    func,
    bounds=(-500, 500),
    population_size=1000,
    survivors=100,
    generations=1000,
    mutation_std=50,
    mutation_decay=0.99,
    no_improve_limit=50,
    restarts=5,
    seed=None
):
    if seed:
        np.random.seed(seed)

    def fitness_fn(x1, x2):
        return -func(x1, x2)      # maximize → minimize func

    all_results = []

    for restart in range(restarts):
        # Fresh population each restart
        solution = [(np.random.uniform(*bounds), np.random.uniform(*bounds))
                    for _ in range(population_size)]
        std = mutation_std
        best_history = []

        for gen in range(generations):
            ranked = sorted(
                [(fitness_fn(s[0], s[1]), s) for s in solution], reverse=True)

            best_fit = ranked[0][0]
            best_x1, best_x2 = ranked[0][1]
            best_history.append(best_fit)

            # Stop condition 1 — stagnation
            if gen > no_improve_limit * 2:
                recent = max(best_history[-no_improve_limit:])
                older  = max(best_history[-no_improve_limit*2:-no_improve_limit])
                if abs(recent - older) < 1e-9:
                    print(f"  [Restart {restart+1}] Stagnated at gen {gen} | "
                          f"x1={best_x1:.4f} x2={best_x2:.4f} f={func(best_x1,best_x2):.6f}")
                    break

            # Stop condition 2 — population collapsed
            if np.std([s[0] for s in solution]) < 1e-6 and \
               np.std([s[1] for s in solution]) < 1e-6:
                print(f"  [Restart {restart+1}] Collapsed at gen {gen} | "
                      f"x1={best_x1:.4f} x2={best_x2:.4f} f={func(best_x1,best_x2):.6f}")
                break

            # Select top survivors
            top      = [s for _, s in ranked[:survivors]]
            e1_pool  = [s[0] for s in top]
            e2_pool  = [s[1] for s in top]

            # Reproduce with elitism + Gaussian mutation
            new_gen = top[:]                      # carry best forward unchanged
            while len(new_gen) < population_size:
                e1 = np.random.choice(e1_pool) + np.random.normal(0, std)
                e2 = np.random.choice(e2_pool) + np.random.normal(0, std)
                new_gen.append((e1, e2))

            solution  = new_gen
            std      *= mutation_decay            # cool down mutation each gen

        all_results.append((best_fit, best_x1, best_x2))

    # Pick best result across all restarts
    best = max(all_results, key=lambda r: r[0])
    return best[1], best[2], func(best[1], best[2])


# ── Run ──
x1, x2, fval = genetic_algorithm(function, seed=42)
print(f"\n🏆 Best: x1={x1:.4f}, x2={x2:.4f}, f={fval:.6f}")

  [Restart 1] Stagnated at gen 124 | x1=14.6683 x2=-7.8003 f=-6.978427
  [Restart 2] Stagnated at gen 117 | x1=15.1239 x2=-8.0980 f=-6.997198
  [Restart 3] Stagnated at gen 101 | x1=14.8837 x2=-7.9310 f=-6.997111
  [Restart 4] Stagnated at gen 119 | x1=14.9329 x2=-7.9427 f=-6.998485
  [Restart 5] Stagnated at gen 135 | x1=15.1506 x2=-8.0957 f=-6.996791

🏆 Best: x1=14.9329, x2=-7.9427, f=-6.998485


In [44]:
y = (14.9329 +-7.9427 - 7)**2 + (2*14.9329 + 3*-7.9427 - 6)**2 - 7
y


-6.99848267